In [1]:
import numpy as np
import os
from typing import List, Dict, Any, Optional
from scripts.parse import opt_geoemtry

top = os.getcwd()
Famide = os.path.join(top, 'F-amide')
Clamide = os.path.join(top, 'Cl-amide')
Bramide = os.path.join(top, 'Br-amide')
Iamide = os.path.join(top, 'I-amide')

paths = [Famide, Clamide, Bramide, Iamide]

In [2]:
OPT_SETTINGS = {
    "JOBTYPE": "opt",
    "METHOD": "wB97M-V",
    "BASIS": "def2-TZVPD",
    "SCF_CONVERGENCE": 8,
    "THRESH": 14,
    "XC_GRID": "000099000590",
    "NL_GRID": 1,
    "SYMMETRY": False,
    "FD_MAT_VEC_PROD": False,
    "MEM_TOTAL": 8000,
    "MEM_STATIC": 1000,
    "SCF_PRINT_FRGM": True
}

EDA_SETTINGS = {
    "JOBTYPE": "eda",
    "EDA2": 1,
    "METHOD": "wB97M-V",
    "BASIS": "def2-TZVPD",
    "SCF_CONVERGENCE": 8,
    "THRESH": 14,
    "XC_GRID": "000099000590",
    "NL_GRID": 1,
    "SYMMETRY": False,
    "EDA_BSSE": False,
    "FD_MAT_VEC_PROD": False,
    "MEM_TOTAL": 32000,
    "MEM_STATIC": 8000,
    "SCF_PRINT_FRGM": True,
    "EDA_CLS_DISP": True
}

def mk_qchem_eda_input(
    atoms: List[List[str]], 
    coords: List[np.ndarray],
    charges: List[int],
    names: List[str],
    fname: Optional[os.PathLike] = None,
    setting: Dict[str, Any] = {}
    ):
    """
    Make a QChem EDA job input file

    Parameters
    ----------
    atoms: List[List[str]]
        Atoms of each fragment
    coords: list of np.ndarray
        Coordinates of each fragment
    charges: list of int
        Net charges of the fragments
    names: list of str
        Names of the fragments
    fname: os.PathLike, optional
        Path to the output qchem job input file
    setting: dict
        Additonal settings that overwrites the `DEFAULT_QCHEM_SETTING`

    Returns
    -------
    inpstr: str
        The content of the input file
    """
    inpstr = [f"$molecule\n{sum(charges)} 1\n"]
    for i in range(len(atoms)):
        inpstr.append(f"-- {names[i]}\n{charges[i]} 1\n")
        for at, crd in zip(atoms[i], coords[i]):
            inpstr.append(f"{at:>3} {crd[0]:>13.7f} {crd[1]:>13.7f} {crd[2]:>13.7f}\n")
    inpstr.append("$end\n\n")
    inpstr.append("$rem\n")
    
    tmpsetting = EDA_SETTINGS.copy()
    tmpsetting.update(setting)
    maxlen = max([len(key) for key in tmpsetting.keys()])
    tmpl = "   {:<" + str(maxlen) + "}  =  {}\n"
    for key, value in tmpsetting.items():
        if isinstance(value, bool): value = str(value).lower()
        inpstr.append(tmpl.format(key.upper(), value))
    inpstr.append("$end\n\n")
    inpstr = "".join(inpstr)
    if fname:
        with open(fname, 'w') as f:
            f.write(inpstr)
    return inpstr


def opt_in(atoms, coords, charge, fname, setting=OPT_SETTINGS, constraint=True):
    inpstr = [f"$molecule\n{charge} 1\n"]
    for atom, coord in zip(atoms, coords):
        inpstr.append(f"{atom:>3} {coord[0]:>13.7f} {coord[1]:>13.7f} {coord[2]:>13.7f}\n")
    inpstr.append("$end\n\n")
    inpstr.append("$rem\n")
    tmpsetting = setting.copy()
    maxlen = max([len(key) for key in tmpsetting.keys()])
    tmpl = "   {:<" + str(maxlen) + "}  =  {}\n"
    for key, value in tmpsetting.items():
        if isinstance(value, bool): value = str(value).lower()
        inpstr.append(tmpl.format(key.upper(), value))
    inpstr.append("$end\n\n")
    if constraint:
        opt_block = (
            "$opt\n"
            "   CONSTRAINT\n"
            f"   stre  1  16  {np.linalg.norm(coords[0] - coords[15])}\n"
            "   ENDCONSTRAINT\n"
            "$end\n\n"
        )
        inpstr.append(opt_block)
    inpstr = "".join(inpstr)
    if fname:
        with open(fname, 'w') as f:
            f.write(inpstr)
    return inpstr

In [3]:
def parse(path):
    with open(path, "r") as f:
        geometry_text = f.read()

    coords = []
    atoms = []
    for line in geometry_text.strip().splitlines():
        parts = line.split()
        if len(parts) == 4 and parts[0].isalpha():
            atom = parts[0]
            x, y, z = map(float, parts[1:])
            atoms.append(atom)
            coords.append([x, y, z])

    coords = np.array(coords)
    return coords, atoms

def rotate(coords, atom1=0, atom2=15):
    coords -= coords[atom1]
    donor = coords[atom2]
    dist = np.linalg.norm(donor)

    v = donor
    v /= np.linalg.norm(v)

    z = np.array([0, 0, 1])

    x = np.cross(v, z)
    x /= np.linalg.norm(x)

    angle = np.arccos(np.dot(v, z))

    K = np.array([[0, -x[2], x[1]],
                    [x[2], 0, -x[0]],
                    [-x[1], x[0], 0]])
    R = np.eye(3) + np.sin(angle) * K + (1 - np.cos(angle)) * (K @ K)

    rotated_coords = coords @ R.T
    rotated_coords[atom2][2] *= dist
    return rotated_coords

def distance(coords, dist):
    dist_init = coords[15, 2] - coords[0, 2]
    shift = dist - dist_init
    coords[12:, 2] += shift
    return coords

# Unopt EDA

Unoptimized structures directly from PDBBind to EDA caluclation

In [ ]:
names = ['fragment1', 'fragment2']

# F-amide complex 
coords, atoms = parse(os.path.join(Famide, 'unopt.txt'))
coords = rotate(coords)
dists = [i/10 for i in range(19, 41)]

atoms = [atoms[:12], atoms[12:]]

for dist in dists:
    coords_dist = distance(coords, dist)
    coords_dists = [coords_dist[:12], coords_dist[12:]]
    charges = [0, 0]
    dist_dir = os.path.join(Famide, f'unopt_eda/eda_{dist:.1f}')
    os.makedirs(dist_dir, exist_ok=True)
    fname = os.path.join(dist_dir, f'eda.in')
    mk_qchem_eda_input(atoms, coords_dists, charges, names, fname)


# Cl-amide complex 
coords, atoms = parse(os.path.join(Clamide, 'unopt.txt'))
coords = rotate(coords)
dists = [i/10 for i in range(19, 41)]

atoms = [atoms[:12], atoms[12:]]

for dist in dists:
    coords_dist = distance(coords, dist)
    coords_dists = [coords_dist[:12], coords_dist[12:]]
    charges = [0, 0]
    dist_dir = os.path.join(Clamide, f'unopt_eda/eda_{dist:.1f}')
    os.makedirs(dist_dir, exist_ok=True)
    fname = os.path.join(dist_dir, f'eda.in')
    mk_qchem_eda_input(atoms, coords_dists, charges, names, fname)


# Br-amide complex 
coords, atoms = parse(os.path.join(Bramide, 'unopt.txt'))
coords = rotate(coords)
dists = [i/10 for i in range(19, 41)]

atoms = [atoms[:12], atoms[12:]]

for dist in dists:
    coords_dist = distance(coords, dist)
    coords_dists = [coords_dist[:12], coords_dist[12:]]
    charges = [0, 0]
    dist_dir = os.path.join(Bramide, f'unopt_eda/eda_{dist:.1f}')
    os.makedirs(dist_dir, exist_ok=True)
    fname = os.path.join(dist_dir, f'eda.in')
    mk_qchem_eda_input(atoms, coords_dists, charges, names, fname)

# I-amide complex 
coords, atoms = parse(os.path.join(Iamide, 'unopt.txt'))
coords = rotate(coords)
dists = [i/10 for i in range(19, 41)]

atoms = [atoms[:12], atoms[12:]]

for dist in dists:
    coords_dist = distance(coords, dist)
    coords_dists = [coords_dist[:12], coords_dist[12:]]
    charges = [0, 0]
    dist_dir = os.path.join(Iamide, f'unopt_eda/eda_{dist:.1f}')
    os.makedirs(dist_dir, exist_ok=True)
    fname = os.path.join(dist_dir, f'eda.in')
    mk_qchem_eda_input(atoms, coords_dists, charges, names, fname)

# Geometry Opt

Geometry optimizaiton of strucuters directly from PDBBind sturctures

In [ ]:
# F-amide
coords, atoms = parse(os.path.join(Famide, 'unopt.txt'))
coords = rotate(coords)
dists = [i/10 for i in range(19, 41)]

for dist in dists:
    coords_dist = distance(coords, dist)
    charges = 0
    dist_dir = os.path.join(Famide, f'opt/opt_{dist:.1f}')
    os.makedirs(dist_dir, exist_ok=True)
    fname = os.path.join(dist_dir, f'opt.in')
    opt_in(atoms, coords_dist, charges, fname)


# Cl-amide
coords, atoms = parse(os.path.join(Clamide, 'unopt.txt'))
coords = rotate(coords)
dists = [i/10 for i in range(19, 41)]

for dist in dists:
    coords_dist = distance(coords, dist)
    charges = 0
    dist_dir = os.path.join(Clamide, f'opt/opt_{dist:.1f}')
    os.makedirs(dist_dir, exist_ok=True)
    fname = os.path.join(dist_dir, f'opt.in')
    opt_in(atoms, coords_dist, charges, fname)


# Br-amide
coords, atoms = parse(os.path.join(Bramide, 'unopt.txt'))
coords = rotate(coords)
dists = [i/10 for i in range(19, 41)]

for dist in dists:
    coords_dist = distance(coords, dist)
    charges = 0
    dist_dir = os.path.join(Bramide, f'opt/opt_{dist:.1f}')
    os.makedirs(dist_dir, exist_ok=True)
    fname = os.path.join(dist_dir, f'opt.in')
    opt_in(atoms, coords_dist, charges, fname)


# I-amide
coords, atoms = parse(os.path.join(Iamide, 'unopt.txt'))
coords = rotate(coords)
dists = [i/10 for i in range(19, 41)]

for dist in dists:
    coords_dist = distance(coords, dist)
    charges = 0
    dist_dir = os.path.join(Iamide, f'opt/opt_{dist:.1f}')
    os.makedirs(dist_dir, exist_ok=True)
    fname = os.path.join(dist_dir, f'opt.in')
    opt_in(atoms, coords_dist, charges, fname)

# Opt EDA (Doesn't converge properly for F-amide, Cl-amide, Br-amide)

EDA of Geometry Opt Structures above

In [ ]:
names = ['fragment1', 'fragment2']
dists = [i/10 for i in range(19, 41)]


# F-amide
for dist in dists:
    path = os.path.join(Famide, f'opt/opt_{dist:.1f}/opt.out')
    atoms, coords = opt_geoemtry(path)
    coords = np.array(coords)
    if len(coords) == 0:
        print(dist)
        print(coords)
    # print(np.linalg.norm(coords[15] - coords[0]))
    # coords = [coords[:12], coords[12:]]
    # atoms = [atoms[:12], atoms[12:]]
    # charges = [0, 0]
    # eda_path = os.path.join(Famide, f'opt_eda/eda_{dist:.1f}')
    # os.makedirs(eda_path, exist_ok=True)
    # fname = os.path.join(eda_path, f'eda.in')
    # mk_qchem_eda_input(atoms, coords, charges, names, fname)


# Cl-amide
for dist in dists:
    path = os.path.join(Clamide, f'opt/opt_{dist:.1f}/opt.out')
    atoms, coords = opt_geoemtry(path)
    coords = np.array(coords)
    if len(coords) == 0:
        print(dist)
        print(coords)
    # print(np.linalg.norm(coords[15] - coords[0]))
    # coords = [coords[:12], coords[12:]]
    # atoms = [atoms[:12], atoms[12:]]
    # charges = [0, 0]
    # eda_path = os.path.join(Famide, f'opt_eda/eda_{dist:.1f}')
    # os.makedirs(eda_path, exist_ok=True)
    # fname = os.path.join(eda_path, f'eda.in')
    # mk_qchem_eda_input(atoms, coords, charges, names, fname)


# Br-amide
for dist in dists:
    path = os.path.join(Bramide, f'opt/opt_{dist:.1f}/opt.out')
    atoms, coords = opt_geoemtry(path)
    coords = np.array(coords)
    if len(coords) == 0:
        print(dist)
        print(coords)
    # print(np.linalg.norm(coords[15] - coords[0]))
    # coords = [coords[:12], coords[12:]]
    # atoms = [atoms[:12], atoms[12:]]
    # charges = [0, 0]
    # eda_path = os.path.join(Famide, f'opt_eda/eda_{dist:.1f}')
    # os.makedirs(eda_path, exist_ok=True)
    # fname = os.path.join(eda_path, f'eda.in')
    # mk_qchem_eda_input(atoms, coords, charges, names, fname)


# I-amide
for dist in dists:
    path = os.path.join(Iamide, f'opt/opt_{dist:.1f}/opt.out')
    atoms, coords = opt_geoemtry(path)
    coords = np.array(coords)
    if len(coords) == 0:
        print(dist)
        print(coords)
    # print(np.linalg.norm(coords[15] - coords[0]))
    # coords = [coords[:12], coords[12:]]
    # atoms = [atoms[:12], atoms[12:]]
    # charges = [0, 0]
    # eda_path = os.path.join(Famide, f'opt_eda/eda_{dist:.1f}')
    # os.makedirs(eda_path, exist_ok=True)
    # fname = os.path.join(eda_path, f'eda.in')
    # mk_qchem_eda_input(atoms, coords, charges, names, fname)

# 1-Opt-EDA (Doesn't Work)

Use Optimized Geometry for TOTAL interaction energy minimum form Unopt-EDA (geom opt converged sucessfully for F-amide and I-amide)

In [ ]:
names = ['fragment1', 'fragment2']

# I-amide
min_dist = 3.0 # minimum energy distance for I-amide from Unopt-EDA
path = os.path.join(Iamide, f'opt/opt_{min_dist:.1f}/opt.out')
atoms, coords = opt_geoemtry(path)
coords = np.array(coords)
coords = rotate(coords)
dists = [i/10 for i in range(19, 41)]

atoms = [atoms[:12], atoms[12:]]

for dist in dists:
    coords_dist = distance(coords, dist)
    coords_dists = [coords_dist[:12], coords_dist[12:]]
    charges = [0, 0]
    eda_path = os.path.join(Iamide, f'1_opt_eda/eda_{dist:.1f}')
    os.makedirs(eda_path, exist_ok=True)
    fname = os.path.join(eda_path, f'eda.in')
    mk_qchem_eda_input(atoms, coords_dists, charges, names, fname)



# F-amide
min_dist = 3.4 # minimum energy distance for F-amide from Unopt-EDA
path = os.path.join(Famide, f'opt/opt_{min_dist:.1f}/opt.out')
atoms, coords = opt_geoemtry(path)
coords = np.array(coords)
coords = rotate(coords)
# print(np.linalg.norm(coords[15] - coords[0]))
dists = [i/10 for i in range(19, 41)]

atoms = [atoms[:12], atoms[12:]]

for dist in dists:
    coords_dist = distance(coords, dist)
    # print(np.linalg.norm(coords[15] - coords[0]))
    coords_dists = [coords_dist[:12], coords_dist[12:]]
    charges = [0, 0]
    eda_path = os.path.join(Famide, f'1_opt_eda/eda_{dist:.1f}')
    os.makedirs(eda_path, exist_ok=True)
    fname = os.path.join(eda_path, f'eda.in')
    mk_qchem_eda_input(atoms, coords_dists, charges, names, fname)

# Iodine Substitution Methods

# I-opt (Doesn't converge for F-amide and Cl-amide)

Geometry Opt with I-amide strucutre with halogen substitution (C-X bond length corrected on fragment 1)

In [ ]:
coords, atoms = parse(os.path.join(Iamide, 'unopt.txt'))
coords = np.array(coords)

Icoord = coords[0]
Cidx = [i for i, e in enumerate(atoms) if e == "C"]
distances = {i: np.linalg.norm(coords[i] - Icoord) for i in Cidx}
Cidx_close = min(distances, key=distances.get)
closest_distance = distances[Cidx_close]
print(Cidx_close)
print(coords[5])
print(np.linalg.norm(coords[5] - Icoord))
print(closest_distance)

5
[28.204 40.094 94.635]
2.0898988013777067
2.0898988013777067


In [ ]:
Icoords, atoms = parse(os.path.join(Iamide, 'unopt.txt'))
Icoords = np.array(Icoords)
charge = 0

# I-amide I-opt
coords = rotate(coords)
atoms[0] = 'I'
path = os.path.join(Iamide, f'I-opt/opt.in')
opt_in(atoms, coords, charge, path, constraint=False)


# F-amide I-opt
coords = rotate(Icoords, 5, 0)
coords[0][2] = 1.35626 # C-F equillibrium distance 
coords = rotate(coords)
coords_dists = distance(coords, 3.0)
atoms[0] = 'F'
path = os.path.join(Famide, f'I-opt/opt.in')
opt_in(atoms, coords, charge, path, constraint=False)


# Cl-amide I-opt
coords = rotate(Icoords, 5, 0)
coords[0][2] = 1.73748 # C-Cl equillibrium distance 
coords = rotate(coords)
coords_dists = distance(coords, 3.0)
atoms[0] = 'Cl'
path = os.path.join(Clamide, f'I-opt/opt.in')
opt_in(atoms, coords, charge, path, constraint=False)


# Br-amide I-opt
coords = rotate(Icoords, 5, 0)
coords[0][2] = 1.89407 # C-Br equillibrium distance 
coords = rotate(coords)
atoms[0] = 'Br'
path = os.path.join(Bramide, f'I-opt/opt.in')
opt_in(atoms, coords, charge, path, constraint=False)

'$molecule\n0 1\n Br     0.0000000     0.0000000     0.0000000\n  C    -0.4565033    -1.4757538    -3.8839498\n  C     0.5056931    -0.7913714    -4.6191640\n  C     1.3762924     0.1333177    -4.0546858\n  C     1.2456737     0.4380464    -2.6436194\n  C     0.2409883    -0.2972212    -1.8550163\n  C    -0.5363373    -1.2347309    -2.5095370\n  H     0.5824386    -0.9694322    -5.6791462\n  H    -1.2591306    -1.7846926    -1.9184384\n  H    -1.0953224    -2.1446057    -4.3417392\n  H     2.1005510     0.5933265    -4.6281538\n  H     1.8429061     1.1539278    -2.2010687\n  C    -0.0538709     2.3430223     3.4684493\n  H    -0.9027802     2.2054882     2.7991024\n  C     0.2818900     1.0140727     4.0200277\n  O    -0.0000000     0.0000000     3.4085610\n  N     0.8436929     0.9593382     5.2087239\n  H     1.1490975     1.7802633     5.7183669\n  C     1.2220189    -0.3915003     5.6462098\n  H     1.6041005    -0.9572417     4.7969582\n  H    -0.3147242     3.0139035     4.26012

# I-Unopt (EDA)

EDA of unoptimized I-amide strucutres with halogen substitution (C-X bond length corrected on fragment 1)

(already done for I-amide in unopt-EDA)

In [10]:
names = ['fragment1', 'fragment2']

Icoords, atoms = parse(os.path.join(Iamide, 'unopt.txt'))
Icoords = np.array(Icoords)
print(np.linalg.norm(Icoords[15] - Icoords[0]))
dists = [i/10 for i in range(19, 41)]

coords = rotate(Icoords, 5, 0)
coords[0][2] = 1.35626
coords = rotate(coords)
atoms = [atoms[:12], atoms[12:]]
atoms[0][0] = 'F'
for dist in dists:
    coords_dists = distance(coords, dist)
    # print(np.linalg.norm(coords[15] - coords[0]))
    coords_dists = [coords_dists[:12], coords_dists[12:]]
    charges = [0, 0]
    eda_path = os.path.join(Famide, f'I-unopt/eda_{dist:.1f}')
    os.makedirs(eda_path, exist_ok=True)
    fname = os.path.join(eda_path, f'eda.in')
    mk_qchem_eda_input(atoms, coords_dists, charges, names, fname)
    
coords = rotate(Icoords, 5, 0)
coords[0][2] = 1.73748
coords = rotate(coords)
atoms[0][0] = 'Cl'
for dist in dists:
    coords_dists = distance(coords, dist)
    # print(np.linalg.norm(coords[15] - coords[0]))
    coords_dists = [coords_dists[:12], coords_dists[12:]]
    charges = [0, 0]
    eda_path = os.path.join(Clamide, f'I-unopt/eda_{dist:.1f}')
    os.makedirs(eda_path, exist_ok=True)
    fname = os.path.join(eda_path, f'eda.in')
    mk_qchem_eda_input(atoms, coords_dists, charges, names, fname)

coords = rotate(Icoords, 5, 0)
coords[0][2] = 1.89407
coords = rotate(coords)
atoms[0][0] = 'Br'
for dist in dists:
    coords_dists = distance(coords, dist)
    # print(np.linalg.norm(coords[15] - coords[0]))
    coords_dists = [coords_dists[:12], coords_dists[12:]]
    charges = [0, 0]
    eda_path = os.path.join(Bramide, f'I-unopt/eda_{dist:.1f}')
    os.makedirs(eda_path, exist_ok=True)
    fname = os.path.join(eda_path, f'eda.in')
    mk_qchem_eda_input(atoms, coords_dists, charges, names, fname)

3.217013211039091


## Long Range

In [ ]:
dists_F = [i/10 for i in range(41, 51)] # 4A - 5A woth 0.1A resolution (searching for minima)
dists = [i/10 for i in range(50, 110, 10)] # long distance for rest
print(dists)

Icoords, atoms = parse(os.path.join(Iamide, 'unopt.txt'))
Icoords = np.array(Icoords)

coords = rotate(Icoords, 5, 0)
coords[0][2] = 1.35626
coords = rotate(coords)
atoms = [atoms[:12], atoms[12:]]
atoms[0][0] = 'F'
for dist in dists_F:
    coords_dists = distance(coords, dist)
    # print(np.linalg.norm(coords[15] - coords[0]))
    coords_dists = [coords_dists[:12], coords_dists[12:]]
    charges = [0, 0]
    eda_path = os.path.join(Famide, f'I-unopt/eda_{dist:.1f}')
    os.makedirs(eda_path, exist_ok=True)
    fname = os.path.join(eda_path, f'eda.in')
    mk_qchem_eda_input(atoms, coords_dists, charges, names, fname)
    


coords = rotate(Icoords, 5, 0)
coords[0][2] = 1.73748
coords = rotate(coords)
atoms[0][0] = 'Cl'
for dist in dists:
    coords_dists = distance(coords, dist)
    # print(np.linalg.norm(coords[15] - coords[0]))
    coords_dists = [coords_dists[:12], coords_dists[12:]]
    charges = [0, 0]
    eda_path = os.path.join(Clamide, f'I-unopt/eda_{dist:.1f}')
    os.makedirs(eda_path, exist_ok=True)
    fname = os.path.join(eda_path, f'eda.in')
    mk_qchem_eda_input(atoms, coords_dists, charges, names, fname)



coords = rotate(Icoords, 5, 0)
coords[0][2] = 1.89407
coords = rotate(coords)
atoms[0][0] = 'Br'
for dist in dists:
    coords_dists = distance(coords, dist)
    # print(np.linalg.norm(coords[15] - coords[0]))
    coords_dists = [coords_dists[:12], coords_dists[12:]]
    charges = [0, 0]
    eda_path = os.path.join(Bramide, f'I-unopt/eda_{dist:.1f}')
    os.makedirs(eda_path, exist_ok=True)
    fname = os.path.join(eda_path, f'eda.in')
    mk_qchem_eda_input(atoms, coords_dists, charges, names, fname)

[5.0, 6.0, 7.0, 8.0, 9.0, 10.0]
